# Lab 03: Tools: Function Calling & MCP

Kiel University · Agentic AI (infAgAI-01a) · Winter 2026

In this lab you wire one calculator tool twice, as the lecture announced: first as raw function calls sent directly to the lab's language model, then behind a minimal MCP-style server in plain Python. Between these two wirings, you give the agent two more tools, one that searches the team's notes and one that reads a note, and build the agent from the lecture's example run: it searches the notes, reads two of them and adds up the costs with the calculator.

**Learning objectives.** After this lab you can:

- declare a tool with a name, a description and a JSON Schema in the Chat Completions format (the format of OpenAI's chat interface for messages and tools, which the lab's provider also uses),
- steer a single request with the **tool-choice mode** (`tool_choice`) and read its effect in `finish_reason` and `tool_calls`,
- **validate tool calls before executing them** (schema, domain rules, permission) and return error messages the model can act on,
- run an agent with three tools and design an **indirect prompt injection** against it,
- build a minimal **MCP-style server and host** in plain Python and add a tool that the host finds through discovery (`tools/list`).

*Requests are limited. Your key allows 200 model requests per hour; one pass through Parts A to F needs about 20 model requests. The provider accepts about 2 requests per 10 seconds, so a cell with several requests may pause for a while; each part says how many requests it makes.*

**Report tasks and hand-in.** Tasks marked 📝 are the report tasks R1 to R3. They come without solutions, and your answers go **directly into this notebook**: each report task is followed by an answer cell marked ✍️ that you fill in (for R2 and R3 the completed code cell is part of your answer). Paste the lines a task asks for into the ✍️ cell as text: the graders do not see cell outputs. There is nothing to upload. The notebook is collected automatically from your folder on the lab server on **Thursday of the week after the lab, at 18:00** (eight days after your lab session); whatever the notebook contains at that moment counts.

## Theory recap: how an agent acts

Everything here is from Session 03. Five ideas carry this lab:

- **A tool call is structured text.** The runtime sends the tool declarations (name, description, JSON Schema for the arguments) along with the messages. If a tool would help, the model writes a structured block with the tool's name and its arguments in JSON. The runtime parses that block, checks it and runs the function. **The model proposes; the runtime decides.**
- **The request-execute-append cycle.** Each round of the loop calls the model once: the model keeps no state between calls, so the runtime sends the whole message list and the tool declarations with every call. `finish_reason` says why the model stopped writing: `"tool_calls"` means the runtime runs the calls and appends one tool message per call, matched to its call by the call's `id`, then sends the whole list again; `"stop"` means a plain-text answer, and the loop ends. One reply may contain several calls (**parallel calls**).
- **Tool-choice modes.** The optional request parameter `tool_choice` decides for one request whether the model may, must or must not call a tool: `auto` (the default if you leave the parameter out: the model decides), `required` (some tool call is forced), a specific tool (that tool is forced), `none` (tools visible, calls forbidden).
- **Failure as observation.** A failed call goes back to the model as a tool result, not as a crash. The runtime validates every call before executing it (schema, domain rules, permission); if a check fails, it tells the model what was wrong and what a valid call looks like.
- **MCP.** The Model Context Protocol standardises where the tools come from. The **host** (the application with the model loop) talks to each **MCP server** through one **client**; an MCP server wraps one system and offers its tools. Client and MCP server exchange small JSON messages: `tools/list` asks which tools exist (**discovery**), `tools/call` runs one.

## Part A: Setup

Same model client as in the earlier labs: `llm.py` in this folder, whose function `llm.chat()` sends your messages to the lab's language model. `import llm` runs the code in `llm.py` once, and its last line prints the provider and the model. The model supports tool calls. The cell imports what this lab needs and checks the connection (1 model request).

In [ ]:
import ast
import json
import operator
from pathlib import Path

import llm

MODEL = llm.MODEL  # name of the lab's language model

try:
    resp = llm.chat([{"role": "user", "content": "Reply with exactly one word: ready"}], model=MODEL)
    print(f"Model '{MODEL}' answered:",
          resp["choices"][0]["message"]["content"].strip())
except Exception as e:
    print("Could not talk to the model:", e)
    print("Check the hint in the message above; without a saved key, run the Onboarding notebook first.")

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `import ast` loads Python's module for **syntax trees**. A syntax tree shows how an expression is built: which operation is applied to which parts. Python builds such a tree for any code it reads, not only for arithmetic: function calls, assignments, loops, whole programs. This lab uses it only for arithmetic expressions. For `(17.5 - 3) * 2` the outermost operation is a multiplication; its left side is the subtraction `17.5 - 3`, its right side is `2`:

  ```
  *   (multiply)
  ├── -   (subtract)
  │   ├── 17.5
  │   └── 3
  └── 2
  ```

  `ast.parse` builds this tree from the string that holds the expression, here `"(17.5 - 3) * 2"`. It only reads the string, it never runs it; that matters because the string comes from the model and could contain any Python code, not just arithmetic. The safe calculator in Part B walks through the tree and computes only parts it allows, such as numbers and `+ - * /`; anything else, such as a function call, it refuses.
- `import operator` loads the arithmetic operations as functions: `operator.add(2, 3)` is `2 + 3`, `operator.mul(2, 3)` is `2 * 3`. The calculator uses them to compute each part of the tree.
- `import json` loads Python's module for JSON, the text format in which the model writes the arguments of a tool call. `json.loads` turns such a text into a dictionary, `json.dumps` turns a Python object into JSON text.
- `from pathlib import Path` imports the class `Path` from Python's module `pathlib`, so the code can write `Path(...)` instead of `pathlib.Path(...)`. `Path` handles file and folder paths: `Path("data/notes")` stands for the folder `data/notes`, and Part E uses three of its functions: `.glob("*.md")` finds the files in that folder whose names end in `.md`, `.read_text()` reads a file, and `.name` gives the file name without the folder.
- `import llm` and `MODEL = llm.MODEL` work as in the earlier labs.
- The `try` block sends one tiny message. If the call succeeds, the reply is printed. If anything goes wrong (no key, no network, an overloaded server), Python jumps to `except Exception as e:`, where `e` holds the error, and prints a hint instead of stopping.
- `resp["choices"][0]["message"]["content"]` is the path to the reply text inside the response dictionary, as in the earlier labs; `.strip()` removes spaces and line breaks at both ends. `print` gets two arguments separated by a comma and prints them with a space between them; the first is an f-string that inserts `MODEL`.

</details>

## Part B: Declaring the calculator

The lecture's calculator takes one string, the whole expression, because that is natural for the model. It puts a burden on the runtime, though: the string needs a safe parser. Python's `eval` is out of the question, because it runs any code it receives, and that is dangerous here: the string comes from the model, and a model that has been misled, for example by instructions hidden in a text it read, could send code that deletes files or reads secrets instead of arithmetic. The cell below is such a safe evaluator. It is given, because this lab is about the tool, not about parsing: it reads the expression as a syntax tree and accepts only numbers, the four basic operations, powers and parentheses. Everything else ends with an error message that says what a valid expression looks like. No model requests in this part.

In [ ]:
OPERATORS = {ast.Add: operator.add, ast.Sub: operator.sub,
             ast.Mult: operator.mul, ast.Div: operator.truediv,
             ast.Pow: operator.pow}

def safe_eval(expression: str) -> float:
    '''Evaluate basic arithmetic: numbers, + - * / **, parentheses. Nothing else.'''
    def evaluate(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value                                   # a number
        if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
            return -evaluate(node.operand)                      # a minus sign, as in -3
        if isinstance(node, ast.BinOp) and type(node.op) in OPERATORS:
            return OPERATORS[type(node.op)](evaluate(node.left), evaluate(node.right))
        raise ValueError("only numbers, + - * / ** and parentheses are allowed; "
                         "write numbers without thousands separators, e.g. '1850 + 2300'")
    try:
        tree = ast.parse(expression, mode="eval")               # exactly one expression
    except SyntaxError:
        raise ValueError("not an arithmetic expression; use plain numbers and operators, "
                         "e.g. '0.15 * 240' instead of '15% of 240'")
    return evaluate(tree.body)

print(safe_eval("(17.5 - 3) * 2"))   # 29.0
print(safe_eval("1850 + 2300"))      # 4150

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `OPERATORS` is a dictionary. Its keys are classes from `ast`: in a syntax tree, `ast.Add` stands for `+`, `ast.Sub` for `-`, `ast.Mult` for `*`, `ast.Div` for `/` and `ast.Pow` for `**`. `ast` has a ready-made class for every element of Python's syntax, for example `ast.Mod` for `%` or `ast.Call` for a function call; `OPERATORS` lists only the five the calculator accepts. These classes only mark which operation stands at a place in the tree; they cannot compute anything. The computing is done by the functions from `operator`, which the dictionary assigns to each marker. Its values are the matching functions from `operator`, written without parentheses: the dictionary only stores which function belongs to which class; `evaluate` calls the function later.
- `def safe_eval(expression: str) -> float:` defines the function `safe_eval` with one parameter, `expression`.
- Inside `safe_eval` a second function, `evaluate(node)`, is defined: a **nested function**, visible only inside `safe_eval`. It is **recursive**, which means it calls itself for the parts of an expression. It walks through the tree it is given, from the top down. For the example tree in Part A, `(17.5 - 3) * 2`, it first evaluates the two sides of the multiplication, the subtraction and the `2`, and then multiplies.
- `evaluate` handles three kinds of node:
  - a **number** (`ast.Constant`): a leaf of the tree; `evaluate` returns its value.
  - a **minus sign** in front of a value (`ast.UnaryOp`, an operation with only one side; its operator `ast.USub` marks the minus sign, like `ast.Sub` marks the minus between two values): `evaluate` calls itself for what follows the sign, usually a number, and returns the result with the opposite sign.
  - an **operation with two sides** (`ast.BinOp`): `evaluate` calls itself for the left and for the right side and combines the two results with the matching function from `OPERATORS`.

  Any other node ends with an error.
- `isinstance(node, ast.Constant)` tests whether `node` is of the given type. In `isinstance(node.value, (int, float))` the inner parentheses build a **tuple** that is passed as one argument; it means "an `int` or a `float`". A constant that is a text, such as `'os'`, fails this test.
- `ast.BinOp` is an operation with two sides, `node.left` and `node.right`. `type(node.op) in OPERATORS` checks whether the operator's class is a key of the dictionary (`in` on a dictionary tests its keys). `OPERATORS[type(node.op)](...)` first looks up the function and then calls it at once with the two evaluated sides.
- Every other kind of node ends at `raise ValueError(...)`: names, function calls such as `__import__('os')`, and also `1,850 + 2,300`, which Python reads as three items separated by commas. The message says what is allowed and gives an example, so a model can fix its next call. The two string pieces inside the parentheses are adjacent string literals, which Python joins into one string.
- `ast.parse(expression, mode="eval")` turns the text into a syntax tree for exactly one expression. A text that is not valid Python raises `SyntaxError`; the `except` turns it into a `ValueError` with a message for the model. `ast.parse` wraps the tree in an outer object; `tree.body` is the top of the actual tree. For `"(17.5 - 3) * 2"` it is the multiplication node from the example in Part A, and `evaluate` starts there.
- There is no `eval` anywhere (Python's built-in function that runs a string as Python code): the code never runs the text, it only reads its structure and computes what it allows.
- The two test lines print 29.0 and 4150. The first result is a `float` because `17.5` is one; the second stays an `int`.

</details>

### The declaration

The cell below is the declaration from the lecture, in the Chat Completions format that the lab's provider expects, and the dispatch table `FUNCS` from the earlier labs. Fill the gaps: the name the model will write, the field every call must contain, and the function behind the name.

In [ ]:
calculator_tool = {
    "type": "function",
    "function": {
        "name": ___,
        "description": (
            "Evaluate a basic arithmetic expression and "
            "return the numeric result. Use this for "
            "arithmetic instead of estimating."),
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "e.g. '(17.4 - 3) * 2'"}},
            "required": [___]}}}

FUNCS = {"calculator": ___}   # dispatch table: tool name -> Python function

print(json.dumps(calculator_tool, indent=2))

<details>
<summary><b>Click here for the solution</b></summary>

```python
calculator_tool = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": (
            "Evaluate a basic arithmetic expression and "
            "return the numeric result. Use this for "
            "arithmetic instead of estimating."),
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "e.g. '(17.4 - 3) * 2'"}},
            "required": ["expression"]}}}

FUNCS = {"calculator": safe_eval}   # dispatch table: tool name -> Python function

print(json.dumps(calculator_tool, indent=2))
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `calculator_tool` is a dictionary nested over several levels, as in the minimal-agent lab: the outer level says `"type": "function"`, and `"function"` holds the three parts of every declaration: `name`, `description` and `parameters`.
- `"name": "calculator"` is the label the model writes when it wants this tool. It must be spelled exactly like the key in `FUNCS`, otherwise the runtime cannot find the function.
- The description is written as three string pieces inside parentheses, which Python joins into one string. It says what the tool does and when to use it. The clause "instead of estimating" changes behaviour: without it, models tend to compute large products in their head and get them wrong. Part G lets you test what a weak description does.
- `"parameters"` is a JSON Schema: the arguments form an object (`"type": "object"`) with one property, `expression`, of type string; its description gives an example value, because the model imitates examples. `"required": ["expression"]` lists the fields every call must contain.
- `FUNCS = {"calculator": safe_eval}` is the dispatch table: the tool name the model uses, mapped to the Python function that runs it. `safe_eval` is written without parentheses: the table stores the function, it does not call it.
- `json.dumps(calculator_tool, indent=2)` turns the dictionary into JSON text, indented by two spaces per level. This is the text that travels to the provider with every request that offers this tool.

</details>

## Part C: Wiring 1: raw function calling

You built `run_tool` and the loop line by line in the minimal-agent lab and used them again in the agent-loop lab. Both come back here as given code, with two changes: `run_agent` receives the tool declarations and the function that executes a tool call as parameters, so that Parts E and F can pass in the ones they need; and there is no log. Read the code and its explanation, then run it.

In [ ]:
SYSTEM_PROMPT = ("You are a careful assistant. Use your tools when they help. "
                 "When you are done, give the final answer as plain text.")

def run_tool(call):
    '''Run one requested call; errors become observations.'''
    try:
        fn = FUNCS[call["function"]["name"]]              # unknown tool -> KeyError
        args = json.loads(call["function"]["arguments"])  # invalid JSON -> error
        result = fn(**args)                               # bad arguments -> TypeError
    except Exception as e:                                # any error becomes text
        result = f"TOOL ERROR: {e!r}"
    return {"role": "tool", "tool_call_id": call["id"], "content": str(result)[:4000]}

def run_agent(task, tools, execute, max_steps=8):
    '''The minimal agent loop. Returns (answer, messages).'''
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": task}]
    for step in range(max_steps):                          # stop: step cap
        resp = llm.chat(messages, model=MODEL, tools=tools)   # REQUEST: messages + tool declarations
        msg = resp["choices"][0]["message"]
        messages.append(msg)
        calls = msg.get("tool_calls") or []
        if not calls:                                      # plain text: the loop ends
            if resp["choices"][0]["finish_reason"] == "length":
                return None, messages                      # cut off at the token limit
            return msg["content"], messages
        for call in calls:                                 # one tool message per call
            print(f"[step {step}] {call['function']['name']}({call['function']['arguments']})")
            messages.append(execute(call))                 # EXECUTE + APPEND
    return None, messages                                  # step cap reached

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `SYSTEM_PROMPT` is the first message of every run. The two string pieces inside the parentheses are joined into one string. It tells the model to use its tools and to end with a plain-text answer.
- `run_tool` is the function from the earlier labs: look the function up by name in `FUNCS`, parse the argument string with `json.loads`, call the function with the arguments unpacked (`fn(**args)` passes the dictionary `{"expression": "2 * 3"}` as `fn(expression="2 * 3")`), and turn any error into a `TOOL ERROR` text. `{e!r}` shows the error with its type, for example `KeyError('doCalc3')`. The return value is the tool message: the role `tool`, the `tool_call_id` that ties the result to the call it answers, and the result as text, cut to 4000 characters (`str(result)[:4000]`).
- `def run_agent(task, tools, execute, max_steps=8):` has two new parameters compared with the earlier labs. `tools` is the list of declarations sent with every request. `execute` is a **runner**: a function that takes one complete tool call from the model (the tool's name and its arguments as a JSON string) and returns the tool message. The runner itself looks up the tool function by name and passes the arguments, as `run_tool` above does. Part E, for example, calls `run_agent(TASK, tools=TOOLS, execute=run_tool_checked)`: `run_tool_checked` is written without parentheses, so `run_agent` receives the function itself, not a result, and calls it as `execute(call)` once per tool call. `max_steps=8` is a **default value**: a caller who leaves it out gets 8.
- `for step in range(max_steps):` runs the body at most `max_steps` times, with `step` counting 0, 1, 2, ... This is the step cap.
- `llm.chat(messages, model=MODEL, tools=tools)` is the REQUEST of the lecture's cycle: the whole message list together with the tool declarations. `resp["choices"][0]["message"]` is the model's reply; it is appended to `messages` unchanged, including its tool calls.
- `calls = msg.get("tool_calls") or []`: `.get` returns `None` when the key is missing (where `msg["tool_calls"]` would raise a `KeyError`), and `or []` replaces `None` by an empty list, so that `if not calls:` and `for call in calls:` work in both cases. An empty list counts as false.
- `if not calls:` decides whether the run ends. The loop goes on as long as the reply contains tool calls, because each call still has to be executed and its result sent back to the model; only a reply without calls is a final answer. The loop does not rely on `finish_reason` for this: when a request names a specific tool, some providers report `"stop"` although the reply contains a call, and the agent still needs that call's result (the test cell on tool-choice modes below may show this). `finish_reason` is still checked once: the value `"length"` marks a reply cut off at the token limit, which `run_agent` returns as `None`.
- `for call in calls:` handles every call of the reply, so parallel calls work: each gets its own tool message before the next request. The `print` line shows step number, tool name and the arguments as the model wrote them; inside the f-string the dictionary keys use single quotes, because the f-string itself is written with double quotes.
- `messages.append(execute(call))` is EXECUTE and APPEND in one line: run the call with whatever function was passed as `execute`, and append the returned tool message.
- Each `return` hands back two values at once, a **tuple**, which the caller unpacks into two names. The last line is reached only when all `max_steps` rounds passed without an answer.

</details>

### Tool-choice modes

`run_agent` never sets `tool_choice`, so our agent always runs in auto. The probe below does not use the agent: it calls `llm.chat` directly and sets `tool_choice` itself, to show what the other modes do. This small test cell sends five requests, one for each pair of prompt and mode in the list `PROBES`: a prompt that needs no tool and a prompt that needs the calculator, each with different modes (5 model requests, about 20 seconds). It prints one line per request. It prints `finish_reason`, the tool calls and the beginning of the text of each reply. Some providers reject a forced mode; the cell then prints the error instead of stopping, and that is a result too. The cell is given.

In [ ]:
GREETING = "Say hello to the lab group in one short sentence."
ARITHMETIC = "What is 987 * 123?"
NAMED = {"type": "function", "function": {"name": "calculator"}}   # a specific tool

PROBES = [(GREETING, None), (GREETING, "required"),
          (ARITHMETIC, None), (ARITHMETIC, "none"), (ARITHMETIC, NAMED)]

for prompt, mode in PROBES:
    options = {} if mode is None else {"tool_choice": mode}   # None: parameter left out
    label = "left out" if mode is None else json.dumps(mode)
    try:
        resp = llm.chat([{"role": "user", "content": prompt}], model=MODEL,
                        tools=[calculator_tool], **options)
    except Exception as e:
        print(f"{prompt[:20]:22} tool_choice {label}: rejected: {str(e)[:100]}")
        continue
    choice = resp["choices"][0]   # model answer, a dictionary
    calls = [(c["function"]["name"], c["function"]["arguments"])
             for c in choice["message"].get("tool_calls") or []]   # list of calls
    text = (choice["message"].get("content") or "")[:50]
    print(f"{prompt[:20]:22} tool_choice {label}: finish_reason {choice['finish_reason']!r}, "
          f"tool_calls {calls}, text {text!r}")

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `NAMED` is the request format for a specific tool: a dictionary that names the function the model must call.
- `PROBES` is a list of **tuples**, each a pair of prompt and mode. `None` stands for "leave the parameter out".
- `for prompt, mode in PROBES:` has two loop variables: each tuple is unpacked, its first element goes into `prompt`, its second into `mode`.
- `options = {} if mode is None else {"tool_choice": mode}` is a **conditional expression**: `x if condition else y`. For `None` it gives an empty dictionary, otherwise a dictionary with the one parameter. `is None` tests for the value `None` itself.
- `**options` in the call spreads the dictionary as named arguments: `llm.chat(..., **{"tool_choice": "none"})` is `llm.chat(..., tool_choice="none")`, and an empty dictionary adds nothing, so the parameter is really left out. `llm.chat` passes such options on to the provider unchanged.
- `label` is only for the printout: `json.dumps(mode)` shows the mode as JSON text, `"required"` with quotes and the named tool as a whole dictionary. `mode` is either a string or, for the named tool, a dictionary; `json.dumps` turns both into text the same way, exactly as they are sent to the provider.
- `try`/`except`: if the provider rejects a mode, the request raises an error; `str(e)[:100]` prints its first 100 characters, and `continue` jumps straight to the next probe.
- `choice = resp["choices"][0]` keeps the first (and only) choice of the reply. `choice["message"]` is a dictionary, not plain text: the provider has already split the model's output into `"content"` (the text) and `"tool_calls"` (a list with one dictionary per call: id, name, arguments). `.get("tool_calls")` returns that list, or `None` if the reply contains no calls. `calls = [... for c in ...]` is a **list comprehension**: it builds the list `calls` with one tuple (tool name, arguments) per tool call in the reply. Read it from the right: `for c in choice["message"].get("tool_calls") or []` introduces `c`, one tool call at a time; `or []` turns a missing list into an empty one. The same as an ordinary loop:

  ```python
  calls = []
  for c in choice["message"].get("tool_calls") or []:
      calls.append((c["function"]["name"], c["function"]["arguments"]))
  ```
- `(choice["message"].get("content") or "")[:50]`: the text may be `None` when the reply is only a tool call; `or ""` replaces it by an empty string, so the slice `[:50]` (the first 50 characters) works.
- In the f-strings, `{prompt[:20]:22}` first cuts the prompt to 20 characters and then pads it to a width of 22, so the columns line up. `{...!r}` prints the value with quotes, so that an empty text shows as `''` and `"stop"` as `'stop'`.
- The `print` gets two f-strings on two lines inside its parentheses; adjacent string literals are joined into one string.

</details>

❓ **Q:** For the arithmetic question in auto mode, the reply contains a tool call but no result. Why does the model not answer with the result?
<details><summary>Click for answer</summary>

Because the probe stops after one request. The model has only asked for the calculator; nothing has run it yet. The probe executes no tool call: it shows only what the model asks for. In the agent, a second round would follow: the runtime runs the tool, appends the result as a tool message, sends the whole list again, and only then does the model write the answer with the number. A tool call is a request to the runtime, not an answer to the user.

</details>

> **📝 Report task R1:** Use the output of the probe. (a) Which mode does the provider apply when `tool_choice` is left out? Show it with the two lines without the parameter. (b) What do `"required"`, `"none"` and the named calculator change compared with leaving the parameter out? Back each answer with the `finish_reason` and `tool_calls` you observed; if the provider rejected a mode, report that instead. (c) Look at the greeting under `"required"`: what did the forced call contain, and what does that tell you about a forced call as evidence?
>
> *No solution is provided. Write your answer in the ✍️ cell below and paste the five printed lines there as text: the graders do not see cell outputs.*

✍️ **Your answer to R1.** Replace this line with your answer (Markdown, (a) to (c) as in the task, with the five probe lines pasted as text).

## Part D: Failure as observation: a validator

`run_tool` already turns every error into an observation, but the observation is a raw Python error such as `TOOL ERROR: TypeError(...)`. The lecture asks more of the runtime: it validates each call before executing it and tells the model what was wrong and what a valid call looks like. The validator below does the schema check (does the call match the declaration?); the domain rules (do the values make sense?) stay in the tool functions, like the error messages of `safe_eval`. No model requests in this cell. Fill the gaps.

In [ ]:
JSON_TYPES = {"string": str, "integer": int, "number": (int, float)}
SCHEMAS = {"calculator": calculator_tool["function"]["parameters"]}   # tool name -> its JSON Schema

def check_call(name, args):
    '''Schema check of one call: a message for the model, or None if the call is valid.'''
    if name not in FUNCS:
        return f"unknown tool '{name}'. Available tools: {list(FUNCS)}."
    schema = SCHEMAS[name]
    for field in schema["required"]:
        if field not in ___:
            return f"missing parameter '{field}'. Required: {schema['required']}."
    for field, prop in schema["properties"].items():
        if field in args and not isinstance(args[field], ___):
            return f"parameter '{field}' must be a {prop['type']}, not {args[field]!r}."
    return None

def run_tool_checked(call):
    '''Validate first, run second; every failure becomes a message the model can act on.'''
    try:
        name, args = call["function"]["name"], json.loads(call["function"]["arguments"])
        problem = ___
        content = f"ERROR: {problem}" if problem else str(FUNCS[name](**args))
    except Exception as e:                       # invalid JSON, or a domain rule of the tool
        content = f"ERROR: {e}"
    return {"role": "tool", "tool_call_id": call["id"], "content": content[:4000]}

print(run_tool_checked({"id": "test", "function": {"name": "calculator", "arguments": '{"expression": 42}'}}))
print(run_tool_checked({"id": "test", "function": {"name": "doCalc3", "arguments": '{"q": "2 * 3"}'}}))

<details>
<summary><b>Click here for the solution</b></summary>

```python
JSON_TYPES = {"string": str, "integer": int, "number": (int, float)}
SCHEMAS = {"calculator": calculator_tool["function"]["parameters"]}   # tool name -> its JSON Schema

def check_call(name, args):
    '''Schema check of one call: a message for the model, or None if the call is valid.'''
    if name not in FUNCS:
        return f"unknown tool '{name}'. Available tools: {list(FUNCS)}."
    schema = SCHEMAS[name]
    for field in schema["required"]:
        if field not in args:
            return f"missing parameter '{field}'. Required: {schema['required']}."
    for field, prop in schema["properties"].items():
        if field in args and not isinstance(args[field], JSON_TYPES[prop["type"]]):
            return f"parameter '{field}' must be a {prop['type']}, not {args[field]!r}."
    return None

def run_tool_checked(call):
    '''Validate first, run second; every failure becomes a message the model can act on.'''
    try:
        name, args = call["function"]["name"], json.loads(call["function"]["arguments"])
        problem = check_call(name, args)
        content = f"ERROR: {problem}" if problem else str(FUNCS[name](**args))
    except Exception as e:                       # invalid JSON, or a domain rule of the tool
        content = f"ERROR: {e}"
    return {"role": "tool", "tool_call_id": call["id"], "content": content[:4000]}

print(run_tool_checked({"id": "test", "function": {"name": "calculator", "arguments": '{"expression": 42}'}}))
print(run_tool_checked({"id": "test", "function": {"name": "doCalc3", "arguments": '{"q": "2 * 3"}'}}))
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `JSON_TYPES` maps the type names of JSON Schema to Python types. A JSON `"number"` may be an integer or a decimal, so its value is the tuple `(int, float)`, which `isinstance` reads as "one of these".
- `SCHEMAS` maps each tool name to the JSON Schema of its parameters, taken from the declaration, so that the check uses exactly what the model was shown.
- `check_call(name, args)` returns a message as soon as it finds a problem; `return` ends the function at once. If all checks pass, it reaches `return None`.
- `if name not in FUNCS:` tests the keys of the dictionary; `list(FUNCS)` turns the keys into a list, so the message names every valid tool.
- The first loop goes through the required fields; `field not in args` is true when the model left one out.
- The second loop walks through the declared parameters: `.items()` yields each name together with its description, and the two loop variables `field` and `prop` receive them. `isinstance(args[field], JSON_TYPES[prop["type"]])` checks the value against the declared type; for `{"expression": 42}` it fails.
- `run_tool_checked(call)` replaces `run_tool`. `name, args = ...` assigns two values in one line: the tool name and the parsed arguments. `problem = check_call(name, args)` runs the schema check.
- `content = f"ERROR: {problem}" if problem else str(FUNCS[name](**args))` is a conditional expression: if there is a problem, the message; otherwise the tool runs, and its result becomes text.
- `try`/`except`: if the arguments are not valid JSON, or the tool itself refuses a value (as `safe_eval` does with `"1,850 + 2,300"`), the error message becomes the observation. The returned tool message has the same shape as the one from `run_tool`.
- The two test lines build calls by hand, as the model would write them, and print the observation the model would get: a wrong type and an unknown tool.

</details>

## Part E: Three tools: search, read, compute

Now the agent from the lecture's example run. The team's notes are in `data/notes/`: two notes on team workshops, each with an invoice, a meeting note that mentions a planned workshop, and a note on an office move. The user asks how much the two team workshops cost in total. The agent gets three tools:

- `search_notes(query, max_results)`: keyword search over the notes; per hit it returns the file name, the title and a one-line summary, not the full note,
- `read_file(file)`: returns one note in full, but only files of the note collection: an **allowlist**, a list of what is permitted, because a file name the model writes is untrusted input,
- the calculator from Part B.

The two cells below are given: the first defines the two functions, the second declares them for the model. No model requests in these cells.

In [ ]:
NOTES = {p.name: p.read_text() for p in Path("data/notes").glob("*.md")}   # file name -> text

def search_notes(query: str, max_results: int = 3) -> str:
    '''Keyword search: file, title and summary line of the best-matching notes.'''
    if not 1 <= max_results <= 5:
        raise ValueError("max_results must be between 1 and 5, e.g. 3")
    hits = []
    for name, text in NOTES.items():
        score = sum(text.lower().count(word) for word in query.lower().split())
        if score > 0:
            title, summary = text.splitlines()[:2]
            hits.append((score, name, title, summary))
    hits.sort(reverse=True)                                      # best score first
    return json.dumps([{"file": name, "title": title, "snippet": summary}
                       for score, name, title, summary in hits[:max_results]])

def read_file(file: str) -> str:
    '''Return one note in full; only files of the note collection are allowed.'''
    if file not in NOTES:                                        # the allowlist
        raise ValueError(f"'{file}' is not a note. Available files: {list(NOTES)}")
    return NOTES[file]

print(search_notes("workshop cost"))

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `NOTES = {p.name: p.read_text() for p in ...}` is a **dictionary comprehension**: for every file `p` that `Path("data/notes").glob("*.md")` finds, the key is the file name and the value is the file's text. The agent works on this dictionary, not on the files.
- `search_notes` has a **default value**, `max_results: int = 3`: a caller who leaves the parameter out gets 3. This is the lecture's point about defaults: only code can fill in a default, a schema can only describe it. The chained comparison `1 <= max_results <= 5` is a domain rule; outside the range the function raises a `ValueError` for the model.
- The loop gives each note a score: `query.lower().split()` splits the query into lower-case words, and `sum(text.lower().count(word) for word in ...)` adds up how often each word occurs in the note.
- `title, summary = text.splitlines()[:2]` takes the first two lines of the note (its title and its summary) and unpacks them into two names.
- Each hit is a tuple `(score, name, title, summary)`. `hits.sort(reverse=True)` sorts tuples by their first element, the score, highest first; for equal scores it compares the file names.
- The `return` line is a list comprehension over the first `max_results` hits; `json.dumps` turns the list into JSON text, short and structured, which is all the model sees of a search.
- `read_file` checks the requested name against the keys of `NOTES` before returning anything: an invented name or a path such as `../secrets.txt` fails and gets a message that lists the valid files.

</details>

In [ ]:
search_notes_tool = {"type": "function", "function": {
    "name": "search_notes",
    "description": ("Search the team's notes by keywords. Returns file, title and a one-line "
                    "summary per hit; use read_file for the full note. Not for arithmetic."),
    "parameters": {"type": "object",
                   "properties": {"query": {"type": "string"},
                                  "max_results": {"type": "integer", "description": "1 to 5, default 3"}},
                   "required": ["query"]}}}

read_file_tool = {"type": "function", "function": {
    "name": "read_file",
    "description": "Return the full text of one note. Use a file name that search_notes returned.",
    "parameters": {"type": "object",
                   "properties": {"file": {"type": "string"}},
                   "required": ["file"]}}}

TOOLS = [calculator_tool, search_notes_tool, read_file_tool]
FUNCS = {"calculator": safe_eval, "search_notes": search_notes, "read_file": read_file}
SCHEMAS = {t["function"]["name"]: t["function"]["parameters"] for t in TOOLS}

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- The two declarations follow the lecture's rules for a strong interface: verb-noun names, descriptions that say what the tool returns, when to use it and when not. `"required": ["query"]` makes `max_results` optional.
- `TOOLS` is the list of declarations the model receives with every request in this part. `FUNCS` and `SCHEMAS` are rebuilt for all three tools; `check_call` looks them up each time it runs, so it now checks all three without any change to its code. In the dictionary comprehension for `SCHEMAS`, `t` is one declaration at a time.

</details>

### The run

The task from the lecture (about 4 model requests, half a minute). Fill the gaps: the declarations to send, and the runner from Part D. Watch the printed calls: if the model reads both workshop notes in one reply, you see two `read_file` lines with the same step number, a parallel call.

In [ ]:
TASK = "How much did the two team workshops cost in total? Use the notes and the calculator."

answer, messages = run_agent(TASK, tools=___, execute=___)
print("\nANSWER:", answer)

<details>
<summary><b>Click here for the solution</b></summary>

```python
TASK = "How much did the two team workshops cost in total? Use the notes and the calculator."

answer, messages = run_agent(TASK, tools=TOOLS, execute=run_tool_checked)
print("\nANSWER:", answer)
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `TASK` is the user's question. It is kept in a name because the attack in R2 reruns the same task.
- `run_agent(TASK, tools=TOOLS, execute=run_tool_checked)` sends all three declarations with every request and executes every call through the validator of Part D.
- A typical trajectory has four requests: `search_notes` with words such as `workshop cost`; `read_file` for the two workshop notes; the calculator with `1850 + 2300`; the answer. The model skips the meeting note because of its summary. Runs differ.

</details>

### An attack on your own agent

Everything a tool returns enters the context, and the model cannot reliably tell content from instructions. Someone who can write into a note can therefore try to steer the agent: this is **indirect prompt injection** (indirect, because the attacker never talks to the model; the text arrives through a tool result). Our agent has no tool that sends or deletes anything, but an attack can still change what the user is told.

> **📝 Report task R2:** Design an indirect prompt injection against the agent of this part. Write the text an attacker would hide in a note into `INJECTION` in the cell below and run it (about 4 model requests): the cell appends your text to the note `workshop_retrospective.md`, in memory only, reruns the task and restores the note. (a) Paste your `INJECTION` text, the printed tool calls and the answer. Did the attack work? If not, change the text once and report both attempts. (b) Name two controls in the runtime, not in the prompt, that would blunt your attack or limit its damage, and say for each where in this notebook's code it would go.
>
> *No solution is provided. Write your answer in the ✍️ cell below; the completed code cell is part of your answer. Paste the printed lines as text: the graders do not see cell outputs.*

In [ ]:
# 📝 Report task R2: write the attacker's text into INJECTION, then run the cell.
INJECTION = ___

original = NOTES["workshop_retrospective.md"]
NOTES["workshop_retrospective.md"] = original + "\n" + INJECTION      # in memory only
try:
    answer, messages = run_agent(TASK, tools=TOOLS, execute=run_tool_checked)
finally:
    NOTES["workshop_retrospective.md"] = original                     # always restore the note
print("\nANSWER:", answer)

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `INJECTION` holds your text as one string. A text over several lines can be written as several string pieces inside parentheses, which Python joins.
- `original = NOTES[...]` keeps the unchanged note. The next line replaces the note in the dictionary by the original plus a line break and your text; the file on disk stays as it is.
- `try`/`finally`: the code under `finally` runs in any case, also when the run fails with an error, so the note is always restored.

</details>

✍️ **Your answer to R2.** Replace this line with your answer (Markdown: (a) your `INJECTION` text, the printed tool calls and the answer, (b) two runtime controls and where each would go).

## Part F: Wiring 2: a minimal MCP-style server and host

The second wiring of the calculator, in plain Python without the MCP SDK, so you see the work the library does. Client and server exchange **JSON-RPC** messages: small dictionaries in a fixed format; a request carries `"jsonrpc": "2.0"`, an `id`, a `method` and its `params`, and the answer carries the same `id` and a `result`. A real server is a separate program; ours is a Python object, so sending a message is a plain method call. Like many existing servers, the lab uses the older protocol revision, in which client and server start with a handshake, `initialize`.

The first cell is the server (no model requests). Fill the gaps: the method name for discovery, and the error flag of a successful call.

In [ ]:
class MiniServer:
    '''A minimal MCP-style server: JSON-RPC requests in, JSON-RPC answers out.'''

    def __init__(self, name):
        self.name = name
        self.tools = {}                          # tool name -> (schema, function)

    def handle(self, request):
        '''Answer one JSON-RPC request.'''
        method, params = request["method"], request.get("params", {})
        if method == "initialize":               # handshake of the older protocol revision
            result = {"protocolVersion": "2025-06-18", "serverInfo": {"name": self.name}}
        elif method == ___:                   # discovery
            result = {"tools": [schema for schema, fn in self.tools.values()]}
        elif method == "tools/call":
            schema, fn = self.tools[params["name"]]
            try:
                text = str(fn(**params["arguments"]))
                result = {"content": [{"type": "text", "text": text}], "isError": ___}
            except Exception as e:                # a failed tool is data, on the wire too
                result = {"content": [{"type": "text", "text": f"ERROR: {e}"}], "isError": True}
        return {"jsonrpc": "2.0", "id": request["id"], "result": result}

calc_server = MiniServer("calc")
calc_server.tools["calculator"] = ({"name": "calculator",
                                    "description": calculator_tool["function"]["description"],
                                    "inputSchema": calculator_tool["function"]["parameters"]},
                                   safe_eval)

<details>
<summary><b>Click here for the solution</b></summary>

```python
class MiniServer:
    '''A minimal MCP-style server: JSON-RPC requests in, JSON-RPC answers out.'''

    def __init__(self, name):
        self.name = name
        self.tools = {}                          # tool name -> (schema, function)

    def handle(self, request):
        '''Answer one JSON-RPC request.'''
        method, params = request["method"], request.get("params", {})
        if method == "initialize":               # handshake of the older protocol revision
            result = {"protocolVersion": "2025-06-18", "serverInfo": {"name": self.name}}
        elif method == "tools/list":                   # discovery
            result = {"tools": [schema for schema, fn in self.tools.values()]}
        elif method == "tools/call":
            schema, fn = self.tools[params["name"]]
            try:
                text = str(fn(**params["arguments"]))
                result = {"content": [{"type": "text", "text": text}], "isError": False}
            except Exception as e:                # a failed tool is data, on the wire too
                result = {"content": [{"type": "text", "text": f"ERROR: {e}"}], "isError": True}
        return {"jsonrpc": "2.0", "id": request["id"], "result": result}

calc_server = MiniServer("calc")
calc_server.tools["calculator"] = ({"name": "calculator",
                                    "description": calculator_tool["function"]["description"],
                                    "inputSchema": calculator_tool["function"]["parameters"]},
                                   safe_eval)
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `class MiniServer:` defines a **class**, a template for objects that bundle data and functions. A function inside a class is a **method**; its first parameter, `self`, is the object the method is called on.
- `__init__` runs when an object is created (`MiniServer("calc")`). It stores the server's name and an empty dictionary of tools in **attributes** of the object (`self.name`, `self.tools`).
- `handle(self, request)` is the server's whole public surface: one JSON-RPC request in, one answer out. `request.get("params", {})` gives an empty dictionary when the request has no parameters.
- `initialize` answers the handshake with the protocol version and the server's name. `tools/list` is discovery: the list comprehension collects the schema of every tool; `for schema, fn in self.tools.values()` unpacks each stored pair, and only `schema` is used.
- `tools/call` looks up the tool by name, calls the function with the arguments, and sends the result back as text inside a `content` list, with `"isError": False`. If the function raises an error, the answer is still a normal result, with the message as text and `"isError": True`: failure as observation, at the level of the protocol.
- The answer copies the request's `id`, so the client can match answer and request.
- The last lines create the server and register the calculator: the schema is written by hand in MCP's format (`name`, `description`, `inputSchema`, MCP's name for `parameters`), built from the parts of the Part B declaration.

</details>

### The host

The host owns the model loop. Its client part is two functions: `connect` performs the handshake and discovery and returns the declarations for the model; `run_tool_mcp` is the runner that passes each tool call to the server with `tools/call`. The loop itself is `run_agent` from Part C, unchanged. The cell is given (no model requests).

In [ ]:
def connect(server):
    '''Handshake and discovery: returns the server's tools as declarations for the model.'''
    info = server.handle({"jsonrpc": "2.0", "id": 1, "method": "initialize"})["result"]
    listed = server.handle({"jsonrpc": "2.0", "id": 2, "method": "tools/list"})["result"]["tools"]
    print(f"connected to '{info['serverInfo']['name']}': tools {[t['name'] for t in listed]}")
    return [{"type": "function", "function": {"name": t["name"], "description": t["description"],
                                              "parameters": t["inputSchema"]}} for t in listed]

def run_tool_mcp(call):
    '''Runner: pass one tool call to the server with tools/call; return the tool message.'''
    request = {"jsonrpc": "2.0", "id": call["id"], "method": "tools/call",
               "params": {"name": call["function"]["name"],
                          "arguments": json.loads(call["function"]["arguments"])}}
    result = calc_server.handle(request)["result"]
    return {"role": "tool", "tool_call_id": call["id"], "content": result["content"][0]["text"]}

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `connect(server)` sends two JSON-RPC requests: `initialize` (the handshake) and `tools/list` (discovery). `["result"]` takes the result out of each answer.
- The `print` line uses a list comprehension, `[t['name'] for t in listed]`, to show the names of the discovered tools; inside the f-string, single quotes mark the keys because the f-string uses double quotes.
- The `return` line translates each MCP schema into the Chat Completions format the model expects: `inputSchema` becomes `parameters`. The host contains no tool names of its own: everything it knows about tools comes from `tools/list`.
- `run_tool_mcp(call)` has the same job and the same return shape as `run_tool` in Part C, so `run_agent` can use it unchanged. It builds a `tools/call` request from the model's call (name and parsed arguments, with the call's `id` as request id), sends it to `calc_server`, and returns the text of the answer as the tool message.

</details>

> **📝 Report task R3:** Add a second tool, `unit_convert`, to `calc_server`. Complete the function in the cell below: it supports at least kg to g, g to kg, km to m, m to km, h to min and min to h, and returns an actionable error for every other pair. Write its schema by hand in MCP's format (`name`, `description`, `inputSchema`), with a description as careful as any tool description: it is one. Then run the cell (about 3 model requests): it connects again and runs a task that needs both tools. (a) Paste the line printed by `connect`, the printed tool calls and the answer. (b) Explain why `connect` and `run_tool_mcp` did not have to change. (c) Compare with the raw wiring of Parts C to E: what would the same extension need there, and what does the MCP wiring cost in return? Name the parts of your own code.
>
> *No solution is provided. Write your answer in the ✍️ cell below; the completed code cell is part of your answer. Paste the printed lines as text: the graders do not see cell outputs.*

In [ ]:
# 📝 Report task R3: complete the function and its schema, then run the cell.
def unit_convert(value: float, from_unit: str, to_unit: str) -> float:
    '''Convert a value from one unit to another.'''
    ___

calc_server.tools["unit_convert"] = (___, unit_convert)

tools = connect(calc_server)                       # discovery finds both tools
answer, messages = run_agent("Convert 2.5 kg to grams, then add 300 g. Use your tools.",
                             tools=tools, execute=run_tool_mcp)
print("\nANSWER:", answer)

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `unit_convert` has three parameters with type hints; the body is yours. A dictionary of conversion factors, keyed by a tuple such as `("kg", "g")`, keeps it short; an unknown pair should raise a `ValueError` whose message lists the supported pairs.
- `calc_server.tools["unit_convert"] = (schema, unit_convert)` registers the function with your schema on the existing server. The schema needs `name`, `description` and `inputSchema` with three properties (`value` of type `number`, `from_unit` and `to_unit` of type `string`) and the list of required fields.
- `connect(calc_server)` performs handshake and discovery again and now finds two tools; `run_agent` runs the task with the discovered declarations and the MCP runner.

</details>

✍️ **Your answer to R3.** Replace this line with your answer (Markdown, (a) to (c) as in the task, with the printed lines pasted as text).

## Wrap-up

**Takeaways**

- A tool call is structured text: the model writes the request; the runtime validates it, runs it and appends the result. The model proposes; the runtime decides.
- `tool_choice` is a setting of one request, not of the model, and a forced call is not an answer backed by evidence.
- Failures and tool results are observations: the runtime returns messages the model can act on, and it treats every tool result as untrusted input.
- MCP standardised where the tools come from, not the loop: the same `run_agent` ran both wirings, and discovery (`tools/list`) let the host use a tool it was never written for.

**Next lab:** prompt engineering for agents: system prompts, few-shot examples and chain-of-thought. The tool descriptions you wrote today were already prompts.

**Before the hand-in (Thursday of next week, 18:00)**

- [ ] **R1** (Part C): the ✍️ cell holds your answers to (a) to (c), with the five probe lines pasted as text.
- [ ] **R2** (Part E): the code cell holds your `INJECTION`; the ✍️ cell holds the printed tool calls and the answer, and two runtime controls with the place in the code where each would go.
- [ ] **R3** (Part F): the code cell holds `unit_convert` and its schema; the ✍️ cell holds the printed lines and your answers to (b) and (c).
- [ ] No gap (`___`) is left in any cell.
- [ ] The notebook is saved in your folder on the lab server. It is collected from there automatically; nothing to upload.